# TestAgent: Zero-Shot and Few-Shot
Run all cells in JupyterLab. Place `input_gt_rule.txt` in the notebook directory. The default input is the expanded 408-rule set (8 targets). The original ZIP contains 167 rules, so it is not silently substituted.

Both modes use the original prompts from `2_validate/1_tc`. Both models receive identical rules and batch boundaries. The few-shot example is retained as in the original prompt, including its SATP example. Batches contain at most 20 rules by default to reduce response-length issues; set `BATCH_SIZE` to a larger value to use one request per target.

The notebook saves raw responses, cleaned test cases, and structural coverage statistics. It does not execute test cases or measure correctness/bugs. Skipped cases are not evidence of correct executable test cases. Transport failures can be retried; outputs are not repaired based on semantic results.

## 1. Models and provider configuration

In [1]:
from pathlib import Path
from openai import OpenAI
from ollama import Client
from getpass import getpass
import hashlib
import json
import os
import re
import time
import yaml

RUN_MODELS = ['gpt-5.6-sol', 'deepseek-v4.1-flash:1017']
REASONING_EFFORT = 'high'
TEMPERATURE = 1
CONTEXT = 2**20
MAX_RETRIES = 5
RETRY_WAIT_SECONDS = 30
RESET_PROGRESS = False

gpt_client = None
ds_client = None

if any(model.startswith('gpt-') for model in RUN_MODELS):
    config = yaml.safe_load((Path.home() / 'cliproxyapi' / 'config.yaml').read_text())
    gpt_client = OpenAI(
        base_url='http://127.0.0.1:8317/v1',
        api_key=config['api-keys'][0],
        timeout=600,
        max_retries=0
    )

if any(not model.startswith('gpt-') for model in RUN_MODELS):
    ollama_api_key = os.environ.get('OLLAMA_API_KEY') or getpass('Ollama API key: ')
    ds_client = Client(host='https://ollama.com', headers={'Authorization': ollama_api_key})

def generate(prompt, model):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            if model.startswith('gpt-'):
                response = gpt_client.chat.completions.create(
                    model=model,
                    reasoning_effort=REASONING_EFFORT,
                    messages=[{'role': 'user', 'content': prompt}]
                )
                result = response.choices[0].message.content
                finish_reason = response.choices[0].finish_reason
            else:
                response = ds_client.chat(
                    model=model,
                    messages=[{'role': 'user', 'content': prompt}],
                    options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}
                )
                result = response.message.content
                finish_reason = getattr(response, 'done_reason', None)
            if not result or not result.strip():
                raise RuntimeError('Empty response')
            return result.strip(), finish_reason
        except Exception as exc:
            print(f'{model}: attempt {attempt}/{MAX_RETRIES}: {type(exc).__name__}')
            if attempt == MAX_RETRIES:
                raise
            time.sleep(RETRY_WAIT_SECONDS)

def safe_name(name):
    return re.sub(r'[^a-zA-Z0-9_.-]+', '_', name)

def atomic_write(path, text):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(text, encoding='utf-8')
    tmp.replace(path)

def cached_generate(prompt, model, directory):
    directory.mkdir(parents=True, exist_ok=True)
    key = hashlib.sha256(json.dumps({
        'model': model,
        'prompt': prompt,
        'reasoning_effort': REASONING_EFFORT,
        'temperature': TEMPERATURE,
        'context': CONTEXT
    }, sort_keys=True).encode()).hexdigest()
    record_path = directory / f'{key}.json'
    if not RESET_PROGRESS and record_path.exists():
        record = json.loads(record_path.read_text(encoding='utf-8'))
        return record['response'], record.get('finish_reason')
    result, finish_reason = generate(prompt, model)
    atomic_write(record_path, json.dumps({
        'model': model,
        'prompt_sha256': hashlib.sha256(prompt.encode()).hexdigest(),
        'response': result,
        'finish_reason': finish_reason
    }, ensure_ascii=False, indent=2))
    return result, finish_reason

## 2. Ground truth and batch configuration

In [2]:
GT_PATH = Path('input_gt_rule.txt')
OUTPUT_DIR = Path('RESULT_TESTAGENT')
MODES = ['zero', 'few']
BATCH_SIZE = 20
EXPECTED_TARGETS = 8
EXPECTED_GT = 408
TARGET_FILTER = None

def load_gt(path):
    targets = {}
    target = None
    for line in path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line:
            continue
        header = re.fullmatch(r'(?:INSTRUCTION|REGISTER):\s*(.+)', line, re.I)
        if header:
            target = header.group(1).strip().lower()
            if target in targets:
                raise ValueError(f'Duplicate target: {target}')
            targets[target] = []
            continue
        if re.match(r'^(?:FORMAT|NUMBER|MODE|WIDTH|FIELDS|EXTENSIONS):', line, re.I):
            continue
        match = re.fullmatch(r'RULE\s+(\d+):\s*(.+)', line)
        if match and target:
            targets[target].append((int(match.group(1)), line))
        else:
            raise ValueError(f'Unexpected line: {line[:200]}')
    if len(targets) != EXPECTED_TARGETS:
        raise ValueError(f'Expected {EXPECTED_TARGETS} targets, got {len(targets)}')
    if sum(map(len, targets.values())) != EXPECTED_GT:
        raise ValueError('Unexpected ground-truth rule count')
    for target, rules in targets.items():
        if not rules or len({number for number, _ in rules}) != len(rules):
            raise ValueError(f'Empty rules or duplicate rule IDs in {target}')
    return targets

targets = load_gt(GT_PATH)

for target, rules in targets.items():
    print(f'{target}: {len(rules)}')

print(f'Total: {sum(map(len, targets.values()))}')

if TARGET_FILTER is not None:
    targets = {target: targets[target] for target in TARGET_FILTER}

assert BATCH_SIZE >= 1

csrrwi: 4
fcvt.w.d: 4
sfence.vma: 14
vle32ff.v: 5
mseccfg: 73
mstatus: 234
satp: 45
scounteren: 29
Total: 408


## 3. Original prompts

In [3]:
PROMPTS = {
    'zero': '''You are an expert RISC-V ISA conformance test generator.

Task:
Given multiple RISC-V architectural rules, generate exactly one minimal RISC-V assembly test case for each rule.

Input:
The input contains multiple rules in this format:

RULE <number>: <architectural rule>

Source restriction:
Use only the given rules as the source of intended architectural behavior.
Do not use external documents.
Do not browse the web.
Do not use known emulator bugs.
Do not infer unrelated architectural behavior.
You may use your pretrained RISC-V knowledge only to write syntactically valid RISC-V assembly and to choose a minimal test sequence that directly exercises the given rule.

Batch requirement:
Treat each rule independently, as if it were the only rule in the input.
If the input contains N rules, output exactly N test cases.
Do not skip any rule.
Do not merge rules.
Do not deduplicate rules.
Do not combine multiple rules into one test case.
Do not let one rule's setup or expected result affect another rule's test case.
Each rule must produce exactly one test case.

Execution assumptions:
Execution starts in M-mode.
The system is RV64.
XLEN = 64.
MXLEN = 64.
SXLEN = 64.
UXLEN = 64.
Most common RISC-V extensions and privilege modes are available unless the rule itself makes the test inapplicable.
A default trap handler is already installed by the test harness.
If a trap occurs, execution stops at the trapping instruction.
If a trap occurs, the harness records mcause.
mcause = 2 means illegal-instruction exception.
Do not write custom trap handlers.
Do not define trap_handler labels.
Do not rely on instructions after a trapping instruction.

Privilege transition helpers:
If a test needs to move from M-mode to S-mode, use:
call machine_to_supervisor
If a test needs to move from M-mode to U-mode, use:
call machine_to_user
If a test needs to move from M-mode to VS-mode, use:
call machine_to_virtual
If a test needs to move from M-mode to VU-mode, use:
call machine_to_virtual_user
If a test needs to move from VS-mode to VU-mode, use:
call vs_to_vu
If a test needs to move from S-mode or HS-mode to U-mode, use:
call supervisor_to_user
Do not define labels with these helper names.
Do not use mret or sret directly unless the rule explicitly requires testing mret or sret itself.

Assembly restrictions:
Output only contiguous RISC-V assembly code and a result line for each rule.
Do not use .globl.
Do not use _start.
Do not use assembler directives.
Do not use .inst, .word, .dword, .byte, .half, .quad, .section, .align, or data labels.
Do not use relocation expressions such as %pcrel_hi, %pcrel_lo, %hi, or %lo.
Do not use ecall, ebreak, or wfi.
Do not generate infinite loops or potentially hanging branches, including:
j .
jal x0, .
beq x0, x0, .
bne x0, x1, .
Do not use andi anywhere.
If a mask is needed, load the mask with li and use register-register and.
Use li for numeric constants and bit-shift expressions.
Do not use la for numeric constants or bit-shift expressions.
Use la only for labels, and avoid labels unless absolutely necessary.

Test-case requirements:
Each test case must be minimal.
Each test case must directly exercise the corresponding rule.
Each test case must use concrete operands and concrete values.
Each test case must produce a deterministic architectural observation.
If the rule requires a trap, place all setup before the trapping instruction and use mcause in the result.
If the rule does not require a trap, do not use mcause as the result.
Prefer checking the directly affected GPR or CSR.
When testing a specific CSR bit or field, check only that bit or field, not the full CSR value.
When testing a read-only-zero, hardwired-zero, or WPRI-zero field, first attempt to write 1 to the relevant bit or field, then read back and check that the relevant bit or field is 0.
When testing WARL behavior, do not assert an illegal readback value. Assert only a deterministic legal observable result if the rule gives one.
When a behavior is optional, permissive, implementation-defined, not directly observable, depends on hidden TLB/cache/pipeline state, or cannot be tested deterministically with standard architectural state, generate a skip test:

nop # Skipped because <short reason>
# Result: x0 = 0

Result-line requirements:
Each test case must end with exactly one # Result line.
The # Result line must describe the final observable architectural state produced by the test code.
The # Result line must be machine-checkable.
The # Result line must not contain prose.
The # Result line must not contain explanations.
The # Result line must not contain parentheses.
The # Result line must not contain OR logic.
Do not use "or", "OR", "||", or alternatives in # Result.
The normal result format is:

# Result: <target> <operator> <value>

<target> may be a general-purpose register, CSR, floating-point register, vector register, or a specific CSR bit or field. <operator> must be either = or !=. <value> must be decimal or hexadecimal.
The only allowed multi-condition result form is two != checks on the same target connected by &&.

Valid examples:

# Result: a0 = 0

# Result: mcause = 2

# Result: mstatus[20] = 1

# Result: sstatus != 0xffffffff && sstatus != 0xffffffffffffffff

Invalid examples:

# Result: the instruction should trap

# Result: a0 = 0 || a0 = 1

# Result: (a0 = 0)

# Result: mstatus is updated correctly

Output format:
Output only the generated test cases.
Do not give an introduction.
Do not give headings.
Do not give explanations.
Do not include markdown.
Do not include bullet points.
Do not include blank lines inside a test case.
Separate test cases by exactly one blank line.

Required output format for each rule:

# Test Case Rule: RULE <number>: <rule text>
##### ASSEMBLY #####
<assembly code>
# Result: <expected architectural result>

Now generate one assembly test case for each rule below:

''',
    'few': '''You are an expert RISC-V ISA conformance test generator.

Task:
Given multiple RISC-V architectural rules, generate exactly one minimal RISC-V assembly test case for each rule.

Input:
The input contains multiple rules in this format:

RULE <number>: <architectural rule>

Source restriction:
Use only the given rules as the source of intended architectural behavior.
Do not use external documents.
Do not browse the web.
Do not use known emulator bugs.
Do not infer unrelated architectural behavior.
You may use your pretrained RISC-V knowledge only to write syntactically valid RISC-V assembly and to choose a minimal test sequence that directly exercises the given rule.

Batch requirement:
Treat each rule independently, as if it were the only rule in the input.
If the input contains N rules, output exactly N test cases.
Do not skip any rule.
Do not merge rules.
Do not deduplicate rules.
Do not combine multiple rules into one test case.
Do not let one rule's setup or expected result affect another rule's test case.
Each rule must produce exactly one test case.

Execution assumptions:
Execution starts in M-mode.
The system is RV64.
XLEN = 64.
MXLEN = 64.
SXLEN = 64.
UXLEN = 64.
Most common RISC-V extensions and privilege modes are available unless the rule itself makes the test inapplicable.
A default trap handler is already installed by the test harness.
If a trap occurs, execution stops at the trapping instruction.
If a trap occurs, the harness records mcause.
mcause = 2 means illegal-instruction exception.
Do not write custom trap handlers.
Do not define trap_handler labels.
Do not rely on instructions after a trapping instruction.

Privilege transition helpers:
If a test needs to move from M-mode to S-mode, use:
call machine_to_supervisor
If a test needs to move from M-mode to U-mode, use:
call machine_to_user
If a test needs to move from M-mode to VS-mode, use:
call machine_to_virtual
If a test needs to move from M-mode to VU-mode, use:
call machine_to_virtual_user
If a test needs to move from VS-mode to VU-mode, use:
call vs_to_vu
If a test needs to move from S-mode or HS-mode to U-mode, use:
call supervisor_to_user
Do not define labels with these helper names.
Do not use mret or sret directly unless the rule explicitly requires testing mret or sret itself.

Assembly restrictions:
Output only contiguous RISC-V assembly code and a result line for each rule.
Do not use .globl.
Do not use _start.
Do not use assembler directives.
Do not use .inst, .word, .dword, .byte, .half, .quad, .section, .align, or data labels.
Do not use relocation expressions such as %pcrel_hi, %pcrel_lo, %hi, or %lo.
Do not use ecall, ebreak, or wfi.
Do not generate infinite loops or potentially hanging branches, including:
j .
jal x0, .
beq x0, x0, .
bne x0, x1, .
Do not use andi anywhere.
If a mask is needed, load the mask with li and use register-register and.
Use li for numeric constants and bit-shift expressions.
Do not use la for numeric constants or bit-shift expressions.
Use la only for labels, and avoid labels unless absolutely necessary.

Test-case requirements:
Each test case must be minimal.
Each test case must directly exercise the corresponding rule.
Each test case must use concrete operands and concrete values.
Each test case must produce a deterministic architectural observation.
If the rule requires a trap, place all setup before the trapping instruction and use mcause in the result.
If the rule does not require a trap, do not use mcause as the result.
Prefer checking the directly affected GPR or CSR.
When testing a specific CSR bit or field, check only that bit or field, not the full CSR value.
When testing a read-only-zero, hardwired-zero, or WPRI-zero field, first attempt to write 1 to the relevant bit or field, then read back and check that the relevant bit or field is 0.
When testing WARL behavior, do not assert an illegal readback value. Assert only a deterministic legal observable result if the rule gives one.
When a behavior is optional, permissive, implementation-defined, not directly observable, depends on hidden TLB/cache/pipeline state, or cannot be tested deterministically with standard architectural state, generate a skip test:

nop # Skipped because <short reason>
# Result: x0 = 0

Result-line requirements:
Each test case must end with exactly one # Result line.
The # Result line must describe the final observable architectural state produced by the test code.
The # Result line must be machine-checkable.
The # Result line must not contain prose.
The # Result line must not contain explanations.
The # Result line must not contain parentheses.
The # Result line must not contain OR logic.
Do not use "or", "OR", "||", or alternatives in # Result.
The normal result format is:

# Result: <target> <operator> <value>

<target> may be a general-purpose register, CSR, floating-point register, vector register, or a specific CSR bit or field. <operator> must be either = or !=. <value> must be decimal or hexadecimal.
The only allowed multi-condition result form is two != checks on the same target connected by &&.

Valid examples:

# Result: a0 = 0

# Result: mcause = 2

# Result: mstatus[20] = 1

# Result: sstatus != 0xffffffff && sstatus != 0xffffffffffffffff

Invalid examples:

# Result: the instruction should trap

# Result: a0 = 0 || a0 = 1

# Result: (a0 = 0)

# Result: mstatus is updated correctly

Few-shot example:

Input:
RULE 54: When the TVM bit in mstatus is 1, attempts to read the satp CSR while executing in S-mode raise an illegal-instruction exception.

Output:

# Test Case Rule: RULE 54: When the TVM bit in mstatus is 1, attempts to read the satp CSR while executing in S-mode raise an illegal-instruction exception.
##### ASSEMBLY #####
li t0, 0x100000
csrs mstatus, t0
call machine_to_supervisor
csrr a0, satp
# Result: mcause = 2

Output format:
Output only the generated test cases.
Do not give an introduction.
Do not give headings.
Do not give explanations.
Do not include markdown.
Do not include bullet points.
Do not include blank lines inside a test case.
Separate test cases by exactly one blank line.

Required output format for each rule:

# Test Case Rule: RULE <number>: <rule text>
##### ASSEMBLY #####
<assembly code>
# Result: <expected architectural result>

Now generate one assembly test case for each rule below:

'''
}

## 4. Generation and structural checks

In [4]:
def clean_test_cases(text):
    text = text.replace('\r\n', '\n').replace('\r', '\n').strip()
    text = re.sub(r'(?im)^\s*#+\s*ASSEMBLY\s*#*\s*$', '##### ASSEMBLY #####', text)
    blocks = []
    current = []
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line or line.startswith('```'):
            continue
        if line.startswith('# Test Case Rule:'):
            if current:
                blocks.append(current)
            current = [line]
        elif current:
            current.append(line)
    if current:
        blocks.append(current)
    return blocks

def summarize_batch(blocks, rules):
    expected = dict(rules)
    counts = {number: 0 for number in expected}
    skipped = 0
    malformed = 0
    unexpected = 0
    header_mismatch = 0
    for block in blocks:
        match = re.fullmatch(r'# Test Case Rule:\s*RULE\s+(\d+):\s*(.+)', block[0])
        if not match:
            malformed += 1
            continue
        number = int(match.group(1))
        if number not in expected:
            unexpected += 1
            continue
        counts[number] += 1
        if block[0] != '# Test Case Rule: ' + expected[number]:
            header_mismatch += 1
        results = [line for line in block if line.startswith('# Result:')]
        if len(results) != 1 or not block[-1].startswith('# Result:') or block.count('##### ASSEMBLY #####') != 1:
            malformed += 1
        elif any(re.match(r'^nop\s*#\s*Skipped\b', line, re.I) for line in block):
            skipped += 1
    return {
        'gt': len(expected),
        'blocks': len(blocks),
        'covered_rule_ids': sum(count > 0 for count in counts.values()),
        'missing_rule_ids': [number for number, count in counts.items() if count == 0],
        'duplicate_rule_ids': [number for number, count in counts.items() if count > 1],
        'unexpected_blocks': unexpected,
        'header_mismatches': header_mismatch,
        'malformed_blocks': malformed,
        'skipped_blocks': skipped
    }

def run_target(model, mode, target, rules):
    directory = OUTPUT_DIR / mode / safe_name(model) / safe_name(target)
    raw = []
    cleaned = []
    reports = []
    for start in range(0, len(rules), BATCH_SIZE):
        batch = rules[start:start + BATCH_SIZE]
        batch_id = start // BATCH_SIZE + 1
        prompt = PROMPTS[mode].rstrip() + '\n\n' + '\n'.join(text for _, text in batch)
        print(f'{mode} | {model} | {target} | batch {batch_id}')
        result, finish_reason = cached_generate(prompt, model, directory / 'CACHE')
        atomic_write(directory / f'RAW_BATCH_{batch_id:03}.txt', result + '\n')
        blocks = clean_test_cases(result)
        raw.append(result)
        cleaned.extend(blocks)
        report = summarize_batch(blocks, batch)
        report['batch'] = batch_id
        report['finish_reason'] = finish_reason
        reports.append(report)
    atomic_write(directory / 'RAW_RESULT.txt', '\n\n'.join(raw) + '\n')
    atomic_write(directory / 'FINAL_RESULT.txt', '\n\n'.join('\n'.join(block) for block in cleaned) + '\n')
    summary = {
        'target': target,
        'gt': len(rules),
        'blocks': sum(report['blocks'] for report in reports),
        'covered_rule_ids': sum(report['covered_rule_ids'] for report in reports),
        'skipped_blocks': sum(report['skipped_blocks'] for report in reports),
        'malformed_blocks': sum(report['malformed_blocks'] for report in reports),
        'unexpected_blocks': sum(report['unexpected_blocks'] for report in reports),
        'header_mismatches': sum(report['header_mismatches'] for report in reports),
        'missing_rule_ids': [number for report in reports for number in report['missing_rule_ids']],
        'duplicate_rule_ids': [number for report in reports for number in report['duplicate_rule_ids']],
        'batches': reports
    }
    atomic_write(directory / 'SUMMARY.json', json.dumps(summary, indent=2))
    return summary

## 5. Run all selected configurations

In [7]:
all_results = {}

for mode in MODES:
    for model in RUN_MODELS:
        summaries = []
        for target, rules in targets.items():
            summaries.append(run_target(model, mode, target, rules))
        directory = OUTPUT_DIR / mode / safe_name(model)
        lines = [
            'Target | GT | Generated | Covered IDs | Skipped | Malformed | Unexpected | Header mismatches | Missing IDs | Duplicate IDs'
        ]
        for s in summaries:
            lines.append(
                f"{s['target']} | {s['gt']} | {s['blocks']} | {s['covered_rule_ids']} | "
                f"{s['skipped_blocks']} | {s['malformed_blocks']} | {s['unexpected_blocks']} | "
                f"{s['header_mismatches']} | {len(s['missing_rule_ids'])} | {len(s['duplicate_rule_ids'])}"
            )
        fields = ['gt', 'blocks', 'covered_rule_ids', 'skipped_blocks', 'malformed_blocks', 'unexpected_blocks', 'header_mismatches']
        totals = [sum(s[field] for s in summaries) for field in fields]
        totals += [sum(len(s[field]) for s in summaries) for field in ['missing_rule_ids', 'duplicate_rule_ids']]
        lines.append('TOTAL | ' + ' | '.join(map(str, totals)))
        atomic_write(directory / 'SUPER_FINAL_RESULT.txt', '\n'.join(lines) + '\n')
        atomic_write(directory / 'SUPER_SUMMARY.json', json.dumps(summaries, indent=2))
        all_results[(mode, model)] = totals
        print('\n'.join(lines))

for (mode, model), totals in all_results.items():
    print(f'{mode} | {model} | GT={totals[0]} | Generated={totals[1]} | Covered IDs={totals[2]}')

zero | gpt-5.6-sol | csrrwi | batch 1
zero | gpt-5.6-sol | fcvt.w.d | batch 1
zero | gpt-5.6-sol | sfence.vma | batch 1
zero | gpt-5.6-sol | vle32ff.v | batch 1
zero | gpt-5.6-sol | mseccfg | batch 1
zero | gpt-5.6-sol | mseccfg | batch 2
zero | gpt-5.6-sol | mseccfg | batch 3
zero | gpt-5.6-sol | mseccfg | batch 4
zero | gpt-5.6-sol | mstatus | batch 1
zero | gpt-5.6-sol | mstatus | batch 2
zero | gpt-5.6-sol | mstatus | batch 3
zero | gpt-5.6-sol | mstatus | batch 4
zero | gpt-5.6-sol | mstatus | batch 5
zero | gpt-5.6-sol | mstatus | batch 6
zero | gpt-5.6-sol | mstatus | batch 7
zero | gpt-5.6-sol | mstatus | batch 8
zero | gpt-5.6-sol | mstatus | batch 9
zero | gpt-5.6-sol | mstatus | batch 10
zero | gpt-5.6-sol | mstatus | batch 11
zero | gpt-5.6-sol | mstatus | batch 12
zero | gpt-5.6-sol | satp | batch 1
zero | gpt-5.6-sol | satp | batch 2
zero | gpt-5.6-sol | satp | batch 3
zero | gpt-5.6-sol | scounteren | batch 1
zero | gpt-5.6-sol | scounteren | batch 2
Target | GT | Genera

In [8]:
from pathlib import Path

root = Path('RESULT_TESTAGENT')
models = ['gpt-5.6-sol', 'deepseek-v4.1-flash_1017']

for mode in ['zero', 'few']:
    for model in models:
        directory = root / mode / model
        sections = []

        for path in sorted(directory.glob('*/FINAL_RESULT.txt')):
            content = path.read_text(encoding='utf-8').strip()
            sections.append(f'===== TARGET: {path.parent.name} =====\n{content}')

        if len(sections) != 8:
            raise ValueError(f'Expected 8 targets, found {len(sections)}: {mode}, {model}')

        output = root / f'TESTAGENT_{mode}_{model}.txt'
        output.write_text('\n\n'.join(sections) + '\n', encoding='utf-8')

        print(f'{output}: {len(sections)} targets')

RESULT_TESTAGENT/TESTAGENT_zero_gpt-5.6-sol.txt: 8 targets
RESULT_TESTAGENT/TESTAGENT_zero_deepseek-v4.1-flash_1017.txt: 8 targets
RESULT_TESTAGENT/TESTAGENT_few_gpt-5.6-sol.txt: 8 targets
RESULT_TESTAGENT/TESTAGENT_few_deepseek-v4.1-flash_1017.txt: 8 targets
